# Online Retail — ETL Assessment

The sections follow the assignment numbering. Run cells **in order the first time**; afterwards you can rerun individual cells to inspect their output. The data is stored in a local SQLite database. Install pandas with `pip install pandas` if needed.

In [2]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [13]:
# Setup — libraries and the explicit CSV location
import pandas as pd
import sqlite3
import hashlib
import shutil
from pathlib import Path
from datetime import datetime

file_path = Path(r"C:\Users\Lenovo\Desktop\Artefact_Assessments\online_retail.csv")
db_path = file_path.parent / "online_retail.db"

print("CSV:", file_path)

CSV: C:\Users\Lenovo\Desktop\Artefact_Assessments\online_retail.csv


## 1. Data Ingestion

In [ ]:
# 1.1. Implement a data ingestion process to fetch the dataset from its source (e.g., CSV file, database).
# Read the dataset from the CSV file on my computer & keep identifiers as strings to preserve their original format.
df = pd.read_csv(file_path, dtype={
    "InvoiceNo": "string", 
    "StockCode": "string",
    "CustomerID": "string"
})
print("Count:", len(df))
df.head()

Count: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [21]:
# 1.2. Load the dataset into a suitable storage system (e.g., relational database, data warehouse)
# Load the original data into a relational database (SQLite), to later compare the source and the cleaned version.

connection = sqlite3.connect(db_path)

df.to_sql("raw_online_retail", connection, if_exists="replace", index=False)

# Read data to make sure it appears in the database.
print("Count:", len(pd.read_sql("SELECT * FROM raw_online_retail", connection)))
display(pd.read_sql("SELECT * FROM raw_online_retail LIMIT 10", connection))

connection.close()

Count: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


## 2. Data Cleaning and Validation

In [ ]:
# 2.1. Handle missing values, duplicates and unusually large quantities.
rows_before = len(df)

df["Description"] = df["Description"].fillna("Unknown product").astype("string").str.strip()
df["Country"] = df["Country"].fillna("Unknown").astype("string").str.strip()
df["CustomerID"] = df["CustomerID"].astype("string").str.replace(r"\.0$", "", regex=True).fillna("UNKNOWN")
df["InvoiceNo"] = df["InvoiceNo"].str.strip()
df["StockCode"] = df["StockCode"].str.strip()

df = df.drop_duplicates().copy()
duplicates_removed = rows_before - len(df)

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

q1, q3 = df["Quantity"].quantile([0.25, 0.75])
iqr = q3 - q1
# Negative quantities can be returns, so they are not removed automatically.
df["QuantityOutlier"] = (df["Quantity"] > q3 + 3 * iqr).astype(int) if len(df) >= 20 and iqr > 0 else 0
print("Duplicate rows removed:", duplicates_removed)
print("Potential quantity outliers:", df["QuantityOutlier"].sum())

In [ ]:
# 2.2. Check mandatory fields and remove invalid records.
invalid = (
    df["InvoiceNo"].isna() | df["StockCode"].isna() |
    df["InvoiceDate"].isna() | df["Quantity"].isna() |
    df["UnitPrice"].isna() | (df["UnitPrice"] < 0) |
    (df["InvoiceNo"] == "") | (df["StockCode"] == "")
)
rejected = df[invalid].copy()
df = df[~invalid].copy()
print("Rejected rows:", len(rejected))
print("Valid rows:", len(df))

## 3. Data Transformation

In [ ]:
# 3.1. Add analytical columns (line sales amount, return indicator and sales date).
df["SalesAmount"] = (df["Quantity"] * df["UnitPrice"]).round(2)
df["IsReturn"] = ((df["Quantity"] < 0) | df["InvoiceNo"].str.startswith("C", na=False)).astype(int)
df["SalesDate"] = df["InvoiceDate"].dt.strftime("%Y-%m-%d")
df[["InvoiceNo", "SalesAmount", "IsReturn", "SalesDate"]].head()

In [ ]:
# 3.2. Check that the columns have suitable data types.
if (df["Quantity"] % 1 == 0).all():
    df["Quantity"] = df["Quantity"].astype("int64")
df["UnitPrice"] = df["UnitPrice"].astype("float64")
print(df.dtypes)

In [ ]:
# 3.3. Standardize text fields.
df["Country"] = df["Country"].str.strip().str.title()
df["Description"] = df["Description"].str.strip().str.upper()
df["CustomerID"] = df["CustomerID"].str.strip()
df.head()

## 4. Data Warehouse Design

In [ ]:
# 4.1. Create tables for products, customers and sales lines.
# 4.2. Link sales to products and customers with foreign keys.
with sqlite3.connect(db_path) as conn:
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS dim_product (
        StockCode TEXT PRIMARY KEY, Description TEXT
    );
    CREATE TABLE IF NOT EXISTS dim_customer (
        CustomerKey TEXT PRIMARY KEY, Country TEXT
    );
    CREATE TABLE IF NOT EXISTS fact_sales (
        SalesLineID INTEGER PRIMARY KEY,
        InvoiceNo TEXT, StockCode TEXT, CustomerKey TEXT,
        InvoiceDate TEXT, SalesDate TEXT,
        Quantity REAL, UnitPrice REAL, SalesAmount REAL,
        IsReturn INTEGER, QuantityOutlier INTEGER,
        FOREIGN KEY (StockCode) REFERENCES dim_product(StockCode),
        FOREIGN KEY (CustomerKey) REFERENCES dim_customer(CustomerKey)
    );
    """)
print("Warehouse tables created.")

## 5. ETL Process

In [ ]:
# 5.1. Load the cleaned data into the dimension and fact tables.
# Replace the existing snapshot to avoid double-counting when this cell is rerun.
products = df[["StockCode", "Description"]].drop_duplicates("StockCode", keep="last")
customers = df[["CustomerID", "Country"]].drop_duplicates("CustomerID", keep="last")
customers = customers.rename(columns={"CustomerID": "CustomerKey"})
sales = df[["InvoiceNo", "StockCode", "CustomerID", "InvoiceDate", "SalesDate",
            "Quantity", "UnitPrice", "SalesAmount", "IsReturn", "QuantityOutlier"]].copy()
sales = sales.rename(columns={"CustomerID": "CustomerKey"})
sales["InvoiceDate"] = sales["InvoiceDate"].dt.strftime("%Y-%m-%d %H:%M:%S")

with sqlite3.connect(db_path) as conn:
    conn.execute("PRAGMA foreign_keys=ON")
    conn.execute("DELETE FROM fact_sales")
    conn.execute("DELETE FROM dim_product")
    conn.execute("DELETE FROM dim_customer")
    products.to_sql("dim_product", conn, if_exists="append", index=False)
    customers.to_sql("dim_customer", conn, if_exists="append", index=False)
    sales.to_sql("fact_sales", conn, if_exists="append", index=False)
print("Sales lines loaded:", len(sales))

### 5.2. Scheduling
Use **Windows Task Scheduler** for daily/hourly runs. A notebook is best for interactive exploration; for unattended scheduling, export it to a `.py` file or use a notebook runner. Running the notebook interactively does **not** create a schedule.

In [ ]:
# 5.2. Display the command to export this notebook to a .py script (run in a terminal).
print('jupyter nbconvert --to script "online_retail_etl_notebook.ipynb"')
print("Then configure the exported script in Windows Task Scheduler.")

## 6. Partitioning and Indexing

In [ ]:
# 6.1. Index columns commonly used for filtering and joining.
# SQLite does not have native table partitioning. A production warehouse could partition by SalesDate/month.
with sqlite3.connect(db_path) as conn:
    conn.executescript("""
    CREATE INDEX IF NOT EXISTS idx_sales_date ON fact_sales(SalesDate);
    CREATE INDEX IF NOT EXISTS idx_sales_product ON fact_sales(StockCode);
    CREATE INDEX IF NOT EXISTS idx_sales_customer ON fact_sales(CustomerKey);
    CREATE INDEX IF NOT EXISTS idx_sales_invoice ON fact_sales(InvoiceNo);
    """)
print("Indexes created.")

## 7. Data Quality Monitoring

In [ ]:
# 7.1. Save one set of data-quality metrics for this run.
run_time = datetime.now().isoformat(timespec="seconds")
with sqlite3.connect(db_path) as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS data_quality_log (
        RunTime TEXT, FileName TEXT, RowsRead INTEGER, RowsLoaded INTEGER,
        DuplicatesRemoved INTEGER, RejectedRows INTEGER, QuantityOutliers INTEGER
    )""")
    conn.execute("INSERT INTO data_quality_log VALUES (?, ?, ?, ?, ?, ?, ?)",
                 (run_time, file_path.name, rows_before, len(df), duplicates_removed,
                  len(rejected), int(df["QuantityOutlier"].sum())))
    print(pd.read_sql_query("SELECT * FROM data_quality_log ORDER BY RunTime DESC LIMIT 5", conn))

In [ ]:
# 7.2. Display simple alerts for detected quality issues.
if len(rejected) > 0:
    print(f"ALERT: {len(rejected)} invalid rows were excluded")
if df.empty:
    print("ALERT: There are no valid sales records")
if df["QuantityOutlier"].sum() > 0:
    print("CHECK: Review unusually large quantities")
if len(rejected) == 0 and not df.empty and df["QuantityOutlier"].sum() == 0:
    print("No quality alerts for this run")

## 8. Data Versioning

In [ ]:
# 8.1. Archive each distinct version of the source CSV using a file hash.
archive_folder = file_path.parent / "archive"
archive_folder.mkdir(exist_ok=True)
file_hash = hashlib.sha256(file_path.read_bytes()).hexdigest()
archive_file = archive_folder / f"online_retail_{file_hash[:12]}.csv"
if not archive_file.exists():
    shutil.copy2(file_path, archive_file)
with sqlite3.connect(db_path) as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS source_versions (
        RunTime TEXT, SourceFile TEXT, SHA256 TEXT, ArchivedFile TEXT
    )""")
    conn.execute("INSERT INTO source_versions VALUES (?, ?, ?, ?)",
                 (datetime.now().isoformat(timespec="seconds"), str(file_path), file_hash, str(archive_file)))
print("Archived file:", archive_file)

## 9. Metadata Management

In [ ]:
# 9.1. Record where tables come from and what transformations were applied.
metadata_rows = [
    ("raw_online_retail", str(file_path), "Original CSV columns", "SQLite: raw_online_retail"),
    ("dim_product", str(file_path), "One description per StockCode", "SQLite: dim_product"),
    ("dim_customer", str(file_path), "One country per CustomerID", "SQLite: dim_customer"),
    ("fact_sales", str(file_path), "Cleaned sales lines; sales amount and return flag", "SQLite: fact_sales")
]
with sqlite3.connect(db_path) as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS metadata (
        TableName TEXT PRIMARY KEY, Source TEXT, Transformation TEXT, Location TEXT
    )""")
    conn.executemany("INSERT OR REPLACE INTO metadata VALUES (?, ?, ?, ?)", metadata_rows)
    print(pd.read_sql_query("SELECT * FROM metadata", conn))

## 10. Performance Optimization

In [ ]:
# 10.1. Inspect the query plan for a date-filtered sales query.
with sqlite3.connect(db_path) as conn:
    plan = conn.execute("""EXPLAIN QUERY PLAN
        SELECT SUM(SalesAmount) FROM fact_sales WHERE SalesDate = '2010-12-01'""").fetchall()
print(plan)

In [ ]:
# 10.2. Cache a frequently used monthly aggregation in a summary table.
with sqlite3.connect(db_path) as conn:
    conn.execute("DROP TABLE IF EXISTS monthly_sales_summary")
    conn.execute("""CREATE TABLE monthly_sales_summary AS
        SELECT substr(SalesDate, 1, 7) AS SalesMonth,
               COUNT(*) AS NumberOfLines,
               ROUND(SUM(SalesAmount), 2) AS TotalSales
        FROM fact_sales GROUP BY substr(SalesDate, 1, 7)""")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_summary_month ON monthly_sales_summary(SalesMonth)")
    print(pd.read_sql_query("SELECT * FROM monthly_sales_summary ORDER BY SalesMonth", conn).head(12))

## Notes and recommendations

- **The first run must be in order:** later cells require the prepared `df` and the warehouse tables created above. Rerunning a cell works after its prerequisites have run; cells are not completely standalone if run out of order.
- **Duplicate records:** exact duplicates are removed here for simplicity, but identical invoice lines might sometimes be legitimate. A real line identifier would help.
- **Customer country:** this keeps the last observed country for each customer; a customer history table would be better if country changes matter.
- **Large datasets:** use an incremental load and a database with native date partitioning instead of rebuilding the entire snapshot.
- **Automation:** this notebook does not run on a schedule until a Windows task or notebook runner is configured.
- **Alerts:** these are visible warnings; email/Teams notifications can be added later.
- **Validation:** run against the full CSV and inspect rejected records and flagged outliers before choosing business thresholds.